# OncoVisionAI - Model Training Notebook

This notebook demonstrates the complete training process for the multimodal cancer detection model.

## Contents
1. Setup and imports
2. Load and prepare data
3. Build multimodal architecture
4. Train the model
5. Evaluate performance
6. Visualize results

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

from models.multimodal_model import MultimodalCancerDetector, create_callbacks
from src.data_preprocessing import CancerDataPreprocessor, MultimodalDataGenerator
from src.utils import print_system_info

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Print system info
print_system_info()

print("\n✓ All imports successful!")

## 2. Load and Prepare Data

In [ ]:
# Configuration
IMAGE_DIR = '../data/raw/images'
CLINICAL_CSV = '../data/clinical_data.csv'
IMG_SIZE = 224
BATCH_SIZE = 32

print("Loading and preparing dataset...")

# Initialize preprocessor
preprocessor = CancerDataPreprocessor(img_size=(IMG_SIZE, IMG_SIZE))

# Prepare multimodal dataset
dataset = preprocessor.prepare_multimodal_dataset(
    image_dir=IMAGE_DIR,
    clinical_csv=CLINICAL_CSV,
    test_size=0.2,
    val_size=0.1
)

print("\n✓ Dataset prepared successfully!")

In [ ]:
# Create data generators
train_generator = MultimodalDataGenerator(
    image_paths=dataset['train']['image_paths'],
    clinical_features=dataset['train']['clinical'],
    labels=dataset['train']['labels'],
    batch_size=BATCH_SIZE,
    img_size=(IMG_SIZE, IMG_SIZE),
    augment=True,
    shuffle=True
)

val_generator = MultimodalDataGenerator(
    image_paths=dataset['val']['image_paths'],
    clinical_features=dataset['val']['clinical'],
    labels=dataset['val']['labels'],
    batch_size=BATCH_SIZE,
    img_size=(IMG_SIZE, IMG_SIZE),
    augment=False,
    shuffle=False
)

test_generator = MultimodalDataGenerator(
    image_paths=dataset['test']['image_paths'],
    clinical_features=dataset['test']['clinical'],
    labels=dataset['test']['labels'],
    batch_size=BATCH_SIZE,
    img_size=(IMG_SIZE, IMG_SIZE),
    augment=False,
    shuffle=False
)

print(f"✓ Data generators created")
print(f"  Training batches: {len(train_generator)}")
print(f"  Validation batches: {len(val_generator)}")
print(f"  Test batches: {len(test_generator)}")

## 3. Build Multimodal Architecture

In [ ]:
# Initialize model
detector = MultimodalCancerDetector(
    img_size=(IMG_SIZE, IMG_SIZE, 3),
    num_clinical_features=5,
    num_classes=2,
    dropout_rate=0.3
)

# Build fusion model
model = detector.build_fusion_model()

# Compile
detector.compile_model(learning_rate=1e-4)

# Show architecture
detector.get_model_summary()

In [ ]:
# Visualize model architecture
keras.utils.plot_model(
    model,
    to_file='../outputs/model_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    dpi=150
)

from IPython.display import Image
Image('../outputs/model_architecture.png')

## 4. Train the Model

In [ ]:
# Create callbacks
callbacks = create_callbacks(
    model_checkpoint_path='../models/checkpoints/best_model.h5',
    tensorboard_log_dir='../logs',
    early_stopping_patience=10
)

print("✓ Callbacks configured")

In [ ]:
# Train model
EPOCHS = 20

print(f"\nStarting training for {EPOCHS} epochs...\n")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training completed!")

## 5. Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].set_title('Model Accuracy', fontweight='bold', fontsize=14)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title('Model Loss', fontweight='bold', fontsize=14)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Validation', linewidth=2)
axes[1, 0].set_title('Model Precision', fontweight='bold', fontsize=14)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Validation', linewidth=2)
axes[1, 1].set_title('Model Recall', fontweight='bold', fontsize=14)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/training_history_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training plots saved")

## 6. Evaluate on Test Set

In [ ]:
# Evaluate
print("Evaluating on test set...\n")

results = model.evaluate(test_generator, verbose=1)

# Print results
print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
for name, value in zip(model.metrics_names, results):
    print(f"{name.capitalize():20s}: {value:.4f}")
print("="*60)

# Calculate F1 score
precision = results[model.metrics_names.index('precision')]
recall = results[model.metrics_names.index('recall')]
f1_score = 2 * (precision * recall) / (precision + recall + 1e-8)
print(f"{'F1-Score':20s}: {f1_score:.4f}")
print("="*60)

## 7. Generate Predictions and Confusion Matrix

In [ ]:
# Get predictions
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = dataset['test']['labels']

print(f"\nPredictions shape: {y_pred_probs.shape}")
print(f"True labels shape: {y_true.shape}")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Benign', 'Malignant'],
    yticklabels=['Benign', 'Malignant'],
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontweight='bold')
plt.xlabel('Predicted Label', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Benign', 'Malignant']))

## 8. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

# ROC curve for malignant class
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/roc_curve_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

## 9. Save Model

In [ ]:
# Save final model
model_path = '../models/saved_models/oncovision_multimodal_notebook.h5'
model.save(model_path)

print(f"✓ Model saved to {model_path}")

# Save model metadata
import json
from datetime import datetime

metadata = {
    'model_name': 'OncoVisionAI Multimodal',
    'architecture': 'MobileNetV3-Small + MLP Fusion',
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'epochs_trained': EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': IMG_SIZE,
    'test_accuracy': float(results[model.metrics_names.index('accuracy')]),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1_score': float(f1_score),
    'auc_score': float(roc_auc),
    'num_parameters': int(model.count_params())
}

with open('../models/model_metadata_notebook.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("✓ Metadata saved")
print("\nModel Summary:")
print(json.dumps(metadata, indent=2))

## 10. Next Steps

Now that the model is trained:

1. **Export to TFLite**: Run `03_gradcam_visualization.ipynb` to see explainability
2. **Generate Grad-CAM**: Visualize model attention
3. **Test on new images**: Use the demo app
4. **Deploy to mobile**: Convert to TFLite and integrate into app

### Commands to run:

```bash
# Export to TFLite
python ../models/export_tflite.py --quantize full

# Launch demo app
streamlit run ../app/demo_app.py
```